# Step 1: Environment Setup & Data Preprocessing

This notebook will guide you through verifying your local or Colab Python environment, downloading the Chest X-ray datasets for Tuberculosis, and loading them into memory for processing.

### 1. Verification of Libraries and CUDA (GPU)
Run the following code block to verify that PyTorch is installed correctly and can see your NVIDIA GPU (if available).

In [ ]:
import sys
import torch
import torchvision
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"OpenCV Version: {cv2.__version__}")
print(f"NumPy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"CUDA Available (GPU Support): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")

### 2. Dataset Information

For Tuberculosis & Pneumonia detection, we use a combined dataset containing:
1. **NORMAL** CXR scans (healthy lung fields)
2. **TUBERCULOSIS** CXR scans (active tuberculosis lesions)
3. **PNEUMONIA** CXR scans (bacterial or viral pneumonia opacities)

#### Dataset Location:
- Downloaded dataset splits and images are located at: `E:\Download\TB Data\data`
- Class directories: `NORMAL`, `TUBERCULOSIS`, `PNEUMONIA` (under `train`, `val`, and `test` splits).

### 3. Load Datasets & Build Metadata CSV
Run this block once you have populated the `data/raw/` directories to generate a metadata CSV containing image paths and their corresponding labels (0 for Normal, 1 for Tuberculosis).

In [ ]:
import os
from pathlib import Path

# Auto-detect dataset directory
if Path('/content/dataset_clean').exists():
    DATA_DIR = Path('/content/dataset_clean')
elif Path('D:/TB N/dataset_clean').exists():
    DATA_DIR = Path('D:/TB N/dataset_clean')
else:
    DATA_DIR = Path('data')

print(f"Dataset root directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    print(f"Using dataset root: {DATA_DIR.resolve()}")


In [ ]:
import pandas as pd
import os
from pathlib import Path

def build_metadata_csv():
    dest_dir = Path("../data") if Path("../data").exists() else Path("data")
    dest_csv = dest_dir / "metadata.csv"
    
    # Flexible paths to try (local Windows and Colab)
    paths_to_try = [
        Path(r"E:\Download\TB Data\data\dataset_splits_with_metadata.csv"),
        Path(r"/content/colab_data/data/dataset_splits_with_metadata.csv"),
        Path(r"/content/colab_data/dataset_splits_with_metadata.csv"),
        dest_dir / "dataset_splits_with_metadata.csv"
    ]
    
    downloaded_csv = None
    for p in paths_to_try:
        if p.exists():
            downloaded_csv = p
            break
            
    if downloaded_csv is None:
        print("[-] Error: Could not find dataset splits CSV in any of the expected paths!")
        for p in paths_to_try:
            print(f"    - {p}")
        return
        
    print(f"[+] Reading downloaded metadata from: {downloaded_csv}")
    df = pd.read_csv(downloaded_csv)
    
    # Filter out UNKNOWN class
    df = df[df['label'] != 'UNKNOWN'].copy()
    
    # Construct relative path
    df['image_path'] = df['split'] + '/' + df['label'] + '/' + df['name']
    
    # Map text labels to numeric labels
    label_map = {
        'NORMAL': 0,
        'TUBERCULOSIS': 1,
        'PNEUMONIA': 2
    }
    df['class_name'] = df['label']
    df['label'] = df['label'].map(label_map)
    
    # Reorder columns
    df = df[['image_path', 'label', 'class_name', 'split', 'source']]
    
    os.makedirs(dest_dir, exist_ok=True)
    df.to_csv(dest_csv, index=False)
    print(f"Successfully generated metadata.csv with {len(df)} images at {dest_csv}")
    print(df["class_name"].value_counts())

build_metadata_csv()

### 4. Plotting Sample Images
Once the metadata file is built, run this block to visualize some sample Chest X-rays from both classes.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image
from pathlib import Path

if Path('/content/dataset_clean/metadata.csv').exists():
    metadata_file = Path('/content/dataset_clean/metadata.csv')
elif Path('../data/metadata.csv').exists():
    metadata_file = Path('../data/metadata.csv')
else:
    metadata_file = Path('data/metadata.csv')

if Path('/content/dataset_clean').exists():
    DATA_ROOT = Path('/content/dataset_clean')
elif Path('D:/TB N/dataset_clean').exists():
    DATA_ROOT = Path('D:/TB N/dataset_clean')
else:
    DATA_ROOT = Path('data')

if metadata_file.exists():
    df = pd.read_csv(metadata_file)
    print(f"Loaded metadata with {len(df):,} total images across {df['class_name'].nunique()} classes.")
    print(df['class_name'].value_counts())
else:
    print('metadata.csv not found. Please verify metadata path.')
